## Data Ingestion

In [1]:
from dotenv import load_dotenv

from typing import TypedDict, List

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq

from langchain_core.documents import Document

from langgraph.graph import StateGraph, END

from ddgs import DDGS

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

##### Load PDF

In [4]:
PDF_PATH = "docs/NexusAI_Corporate_Policy_Handbook.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("Pages:", len(documents))

Pages: 44


##### Chunking

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 85


##### Embeddings

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

d:\RAG\rag_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6664.86it/s]


##### Vecto Store

In [7]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


##### Reload Existing DB

In [8]:
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


##### Retriever

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

##### Graph State

In [10]:
class GraphState(TypedDict):

    query: str

    retrieve_flag: bool

    documents: List[Document]

    context: str

    answer: str

    supported: str

    utility_score: int

##### Retriever Token

In [11]:
def decide_retrieval(state):

    result = llm.invoke(
        f"""
Question:
{state['query']}

Should retrieval be used?

Answer only:
Yes or No
"""
    ).content

    return {
        "retrieve_flag":
        "yes" in result.lower()
    }

##### Direct Generation

In [12]:
def generate_direct(state):

    answer = llm.invoke(
        state["query"]
    ).content

    return {
        "answer": answer
    }

##### Retrieval

In [13]:
def retrieve(state):

    docs = retriever.invoke(
        state["query"]
    )

    return {
        "documents": docs
    }

##### In the Self-Reflective Retrieval-Augmented Generation

In [14]:
def is_relevant(state):

    relevant_docs = []

    for doc in state["documents"]:

        prompt = f"""
Question:
{state['query']}

Document:
{doc.page_content}

Is this document relevant to answer the question?

Answer only:
YES
or
NO
"""

        result = llm.invoke(prompt).content

        if "yes" in result.lower():
            relevant_docs.append(doc)

    print(f"\n[RELEVANT DOCS]: {len(relevant_docs)}")

    return {
        "documents": relevant_docs
    }

##### Rewrite Query

In [15]:
def rewrite_query(state):

    query = state["query"]

    prompt = f"""
Rewrite the question for better web search.

Question:
{query}

Only return rewritten query.
"""

    new_query = llm.invoke(prompt).content

    print("\n[REWRITTEN QUERY]")
    print(new_query)

    return {
        "query": new_query
    }

##### Web Search

In [16]:
from ddgs import DDGS
from langchain_core.documents import Document

def web_search(state):

    query = state["query"]

    results = []

    with DDGS() as ddgs:
        search_results = ddgs.text(
            query,
            max_results=5
        )

        for r in search_results:
            results.append(
                Document(
                    page_content=r["body"],
                    metadata={
                        "title": r["title"],
                        "url": r["href"]
                    }
                )
            )

    print(f"\n[WEB RESULTS]: {len(results)}")

    return {
        "documents": results
    }

##### Generate From Context

In [17]:
def generate_from_context(state):

    context = "\n\n".join(
        doc.page_content
        for doc in state["documents"]
    )

    answer = llm.invoke(
        f"""
Answer only from context.

Context:
{context}

Question:
{state['query']}
"""
    ).content

    return {
        "answer": answer,
        "context": context
    }

##### Is Support

In [18]:
def support_check(state):

    result = llm.invoke(
        f"""
Context:
{state['context']}

Answer:
{state['answer']}

Supported?

Answer:
yes
partially
no
"""
    ).content

    return {
        "supported":
        result.lower()
    }

##### Is Useful

In [19]:
def utility_check(state):

    result = llm.invoke(
        f"""
Question:
{state['query']}

Answer:
{state['answer']}

Rate usefulness from 1-5.

Return number only.
"""
    ).content

    try:
        score = int(result)
    except:
        score = 1

    return {
        "utility_score": score
    }

##### Routers

In [20]:
def retrieve_router(state):

    if state["retrieve_flag"]:
        return "retrieve"

    return "direct"

In [21]:
def relevance_router(state):

    if len(state["documents"]) > 0:
        return "generate"

    return "rewrite"

In [22]:
def support_router(state):

    if "yes" in state["supported"]:
        return "utility"

    return "rewrite"

##### Build Graph

In [23]:
builder = StateGraph(GraphState)

builder.add_node(
    "decide_retrieval",
    decide_retrieval
)

builder.add_node(
    "generate_direct",
    generate_direct
)

builder.add_node(
    "retrieval_node",
    retrieve
)

builder.add_node(
    "is_relevant",
    is_relevant
)

builder.add_node(
    "rewrite_query",
    rewrite_query
)

builder.add_node(
    "web_search",
    web_search
)

builder.add_node(
    "generate_from_context",
    generate_from_context
)

builder.add_node(
    "support_check",
    support_check
)

builder.add_node(
    "utility_check",
    utility_check
)

builder.set_entry_point(
    "decide_retrieval"
)

builder.add_conditional_edges(
    "decide_retrieval",
    retrieve_router,
    {
        "retrieve": "retrieval_node",
        "direct": "generate_direct"
    }
)

builder.add_edge(
    "retrieval_node",
    "is_relevant"
)

builder.add_conditional_edges(
    "is_relevant",
    relevance_router,
    {
        "generate": "generate_from_context",
        "rewrite": "rewrite_query"
    }
)

builder.add_edge(
    "rewrite_query",
    "web_search"
)

builder.add_edge(
    "web_search",
    "generate_from_context"
)

builder.add_edge(
    "generate_from_context",
    "support_check"
)

builder.add_conditional_edges(
    "support_check",
    support_router,
    {
        "utility": "utility_check",
        "rewrite": "rewrite_query"
    }
)

builder.add_edge(
    "utility_check",
    END
)

builder.add_edge(
    "generate_direct",
    END
)

graph = builder.compile()

In [26]:
query = input("Question: ")

result = graph.invoke(
    {
        "query": query
    }
)

print("\nQuery:\n")
print(query)

print("\nAnswer:\n")
print(result["answer"])

print("\nSupport:")
print(result.get("supported"))

print("\nUtility:")
print(result.get("utility_score"))


[RELEVANT DOCS]: 5

Query:

What is Deep Learning?

Answer:

Deep Learning, also referred to as Deep Structured Learning, is a part of Machine Learning that takes it a lot closer to Artificial Intelligence. It creates better models and representations to learn from large-scale unlabeled data by using architectures that contain layers of neurons. It provides semi-supervised or unsupervised feature learning algorithms and hierarchical feature extraction, and can automatically extract complex representations from a huge amount of unsupervised data.

Support:
the answer is: yes

the provided text supports the definition of deep learning as a part of machine learning that uses architectures with layers of neurons to learn from large-scale unlabeled data, and provides semi-supervised or unsupervised feature learning algorithms and hierarchical feature extraction. the text also mentions that deep learning can automatically extract complex representations from a huge amount of unsupervised da